In [ ]:
!pip install -q langchain langchain-groq

In [ ]:
!pip install -q openai

In [ ]:
# once per Colab session

from google.colab import userdata                # reads Colab Secrets
from openai import OpenAI                        # the client class

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),        # who you are, who pays
    base_url="https://api.groq.com/openai/v1",   # send requests to Groq, not OpenAI
)

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",                  # which model answers
    messages=[
        {"role": "system", "content": "You are a funny joker"},
        {"role": "user",   "content": "tell me a joke"},
    ],
)
print(response.choices[0].message.content)

Why don’t scientists trust atoms anymore?

Because they **make up** everything! 😄


In [ ]:
SYSTEM = """ROLE:   You are the assistant for Sweet Crumbs Home Bakery.
FACTS:  We bake to order. Orders need 24 hours' notice. All cakes have eggless versions.
        Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80.
RULES:  Never promise a delivery time. If the answer isn't in FACTS, say "I'll check with the owner."
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    temperature=0.2,                              # facts and prices: keep it focused
    messages=[{"role": "system", "content": SYSTEM},
              {"role": "user",   "content": "Can I get an eggless cake today evening?"}],
)
print(response.choices[0].message.content)
print(response.usage)

We’d love to make an egg‑less cake for you! All our cakes can be egg‑less, but we need at least 24 hours notice for any order. Please let us know the flavor you’d like, and we’ll schedule it.
CompletionUsage(completion_tokens=297, prompt_tokens=183, total_tokens=480, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=236, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.155749679, prompt_time=0.012616008, completion_time=0.622592423, total_time=0.635208431)


In [ ]:
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}   # key → value

In [ ]:

MENU.get("donut", "donuts not available")   # 'not on menu'  (no crash)

'donuts not available'

In [ ]:
order = {"name": "Ravi", "item": "plum cake", "qty": 2, "eggless": True}


In [ ]:
order["qty"] * MENU[order["item"]] # 1000


1000

In [ ]:
import json
text = json.dumps(order)           # dict → JSON string, to send or save
back = json.loads(text)

In [ ]:
text

'{"name": "Ravi", "item": "plum cake", "qty": 2, "eggless": true}'

In [ ]:
back

{'name': 'Ravi', 'item': 'plum cake', 'qty': 2, 'eggless': True}

In [ ]:
def get_price(item: str, qty: int = 1) -> int:
    return MENU[item.lower().strip()] * qty

get_price("cupcake", 6)    # 480
get_price("Plum Cake")

500

In [ ]:
def get_price(item, qty=1) :
    """Return the total price in rupees for qty of an item on the Sweet Crumbs menu."""
    return MENU[item.lower().strip()] * qty

get_price("cupcake", 6)    # 480
get_price("Plum Cake")

500

In [ ]:
try:
  a=10
  print(a+b)
  print('hi')
except:
  print('b is not defined')


b is not defined


In [ ]:
import json, functools
from openai import RateLimitError, APIConnectionError

MODEL = "openai/gpt-oss-120b"
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

SYSTEM = f"""ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Menu with prices in rupees: {json.dumps(MENU)}
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time. If it isn't in FACTS, say "I'll check with the owner."
        Remember the customer's name and order during the chat.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty

history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:]
    try:
        response = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.3)
        reply = response.choices[0].message.content
    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."
    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."
    history.append({"role": "assistant", "content": reply})
    return reply


while True:
    text = input("You: ")
    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Sweet Crumbs!")
        break
    print("Bot:", chat(text))

You: cupcakes
Bot: Sure! Our cupcakes are ₹80 each. Let me know how many you’d like and if you prefer eggless. Please note we need 24‑hour notice. Thanks!
You: quit
Bot: Thank you for visiting Sweet Crumbs!


In [ ]:
history

[{'role': 'system',
  'content': 'ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.\nFACTS:  Menu with prices in rupees: {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}\n        Orders need 24 hours\' notice. All cakes have eggless versions.\nRULES:  Never promise a delivery time. If it isn\'t in FACTS, say "I\'ll check with the owner."\n        Remember the customer\'s name and order during the chat.\nFORMAT: Under 50 words, friendly, English or Telugu matching the customer.'},
 {'role': 'user', 'content': 'hi'},
 {'role': 'assistant',
  'content': 'Hi! 😊 Welcome to Sweet Crumbs Home Bakery. How can I help you today? We have plum cake (₹500), chocolate cake (₹650) and cupcakes (₹80). Let me know what you’d like!'},
 {'role': 'user', 'content': 'tea vundha'},
 {'role': 'assistant',
  'content': 'టీ ఉందా?\u202fనేను యజమానితో చెక్ చేసి మీకు తెలియజేస్తాను!'},
 {'role': 'user', 'content': 'tell'},
 {'role': 'assistant',
  'content': "I'll check with the owner and